# PySpark RDD Data Engineering Project — Banking Analytics

This project uses two related input files and focuses on practical PySpark RDD transformations, joins, aggregations, filtering, sorting, deduplication, and business analysis.

**Input files:** `customers.csv`, `accounts.csv`

**Total scenario questions:** 30

In [ ]:
from pyspark import SparkContext
import csv

sc = SparkContext.getOrCreate()

customer = sc.textFile("customers.csv").mapPartitions(lambda it: csv.reader(it)).filter(lambda x: x[0]!="customer_id")
account = sc.textFile("accounts.csv").mapPartitions(lambda it: csv.reader(it)).filter(lambda x: x[0]!="account_id")


## Q1. High-Value Transactions

Find transactions above ₹50,000 for Premium or Business customers.

In [ ]:
c=customer.map(lambda x:(x[0],x[2],x[3],x[4]))
a=account.map(lambda x:(x[1],x[5],x[3],x[4]))
r=a.join(c).filter(lambda x: float(x[1][0])>50000 and x[1][1][1] in ["Premium","Business"])
print(r.collect())

## Q2. City-Wise Transaction Revenue

Calculate total transaction amount for each city and show cities above ₹1,000,000.

In [ ]:
c=customer.map(lambda x:(x[0],x[2]))
a=account.map(lambda x:(x[1],float(x[5])))
r=a.join(c).map(lambda x:(x[1][1],x[1][0])).reduceByKey(lambda a,b:a+b).filter(lambda x:x[1]>1000000)
print(r.sortBy(lambda x:x[1],ascending=False).collect())

## Q3. Risk-Wise Exposure

Calculate total account balance by risk category.

In [ ]:
c=customer.map(lambda x:(x[0],x[4]))
a=account.map(lambda x:(x[1],float(x[6])))
print(a.join(c).map(lambda x:(x[1][1],x[1][0])).reduceByKey(lambda a,b:a+b).collect())

## Q4. Account Type Count

Count transactions for every account type.

In [ ]:
a=account.map(lambda x:(x[2],1)); print(a.reduceByKey(lambda a,b:a+b).collect())

## Q5. Transaction Mode Count

Count transactions by transaction mode.

In [ ]:
a=account.map(lambda x:(x[4],1)); print(a.reduceByKey(lambda a,b:a+b).collect())

## Q6. Average Amount

Calculate average transaction amount by transaction type.

In [ ]:
a=account.map(lambda x:(x[3],(float(x[5]),1))); r=a.reduceByKey(lambda a,b:(a[0]+b[0],a[1]+b[1])).mapValues(lambda x:x[0]/x[1]); print(r.collect())

## Q7. Maximum Transaction

Find the maximum transaction for each transaction type.

In [ ]:
a=account.map(lambda x:(x[3],float(x[5]))); print(a.reduceByKey(max).collect())

## Q8. Minimum Transaction

Find the minimum transaction for each transaction type.

In [ ]:
a=account.map(lambda x:(x[3],float(x[5]))); print(a.reduceByKey(min).collect())

## Q9. Premium Customer Count

Count Premium customers by city.

In [ ]:
c=customer.filter(lambda x:x[3]=='Premium').map(lambda x:(x[2],1)); print(c.reduceByKey(lambda a,b:a+b).collect())

## Q10. High Risk Count

Count High-risk customers by city.

In [ ]:
c=customer.filter(lambda x:x[4]=='High').map(lambda x:(x[2],1)); print(c.reduceByKey(lambda a,b:a+b).collect())

## Q11. Deposit Analysis

Calculate total deposits by city.

In [ ]:
c=customer.map(lambda x:(x[0],x[2])); a=account.filter(lambda x:x[3]=='Deposit').map(lambda x:(x[1],float(x[5]))); print(a.join(c).map(lambda x:(x[1][1],x[1][0])).reduceByKey(lambda a,b:a+b).collect())

## Q12. Withdrawal Analysis

Calculate total withdrawals by customer.

In [ ]:
a=account.filter(lambda x:x[3]=='Withdrawal').map(lambda x:(x[1],float(x[5]))); print(a.reduceByKey(lambda a,b:a+b).sortBy(lambda x:x[1],ascending=False).take(10))

## Q13. Business Customers

List Business customers with their total transaction amount.

In [ ]:
c=customer.filter(lambda x:x[3]=='Business').map(lambda x:(x[0],x[1])); a=account.map(lambda x:(x[1],float(x[5]))); print(a.reduceByKey(lambda a,b:a+b).join(c).sortBy(lambda x:x[1][0],ascending=False).collect())

## Q14. Balance Above Threshold

Find accounts with balance above ₹250,000.

In [ ]:
print(account.filter(lambda x:float(x[6])>250000).map(lambda x:(x[0],x[1],float(x[6]))).collect())

## Q15. UPI High Value

Find UPI transactions above ₹20,000.

In [ ]:
print(account.filter(lambda x:x[4]=='UPI' and float(x[5])>20000).collect())

## Q16. NEFT Total

Calculate total NEFT transaction amount.

In [ ]:
print(account.filter(lambda x:x[4]=='NEFT').map(lambda x:float(x[5])).sum())

## Q17. IMPS Average

Calculate average IMPS transaction amount.

In [ ]:
r=account.filter(lambda x:x[4]=='IMPS').map(lambda x:(float(x[5]),1)).reduce(lambda a,b:(a[0]+b[0],a[1]+b[1])); print(r[0]/r[1])

## Q18. Customer Transaction Count

Calculate number of transactions per customer.

In [ ]:
r=account.map(lambda x:(x[1],1)).reduceByKey(lambda a,b:a+b); print(r.sortBy(lambda x:x[1],ascending=False).take(20))

## Q19. Customer Total Spend

Calculate total transaction amount per customer.

In [ ]:
r=account.map(lambda x:(x[1],float(x[5]))).reduceByKey(lambda a,b:a+b); print(r.sortBy(lambda x:x[1],ascending=False).take(20))

## Q20. City Premium Exposure

Calculate Premium customer transaction exposure by city.

In [ ]:
c=customer.filter(lambda x:x[3]=='Premium').map(lambda x:(x[0],x[2])); a=account.map(lambda x:(x[1],float(x[5]))); print(a.join(c).map(lambda x:(x[1][1],x[1][0])).reduceByKey(lambda a,b:a+b).collect())

## Q21. Savings Total

Calculate total transaction value for Savings accounts.

In [ ]:
print(account.filter(lambda x:x[2]=='Savings').map(lambda x:float(x[5])).sum())

## Q22. Current Account Count

Count Current account transactions by mode.

In [ ]:
print(account.filter(lambda x:x[2]=='Current').map(lambda x:(x[4],1)).reduceByKey(lambda a,b:a+b).collect())

## Q23. Risk Transaction Average

Find average transaction amount for each risk category.

In [ ]:
c=customer.map(lambda x:(x[0],x[4])); a=account.map(lambda x:(x[1],float(x[5]))); r=a.join(c).map(lambda x:(x[1][1],(x[1][0],1))).reduceByKey(lambda a,b:(a[0]+b[0],a[1]+b[1])).mapValues(lambda x:x[0]/x[1]); print(r.collect())

## Q24. Top 10 Transactions

Find the ten largest transactions.

In [ ]:
print(account.map(lambda x:(x[0],float(x[5]))).sortBy(lambda x:x[1],ascending=False).take(10))

## Q25. Small Transactions

Count transactions below ₹1,000 by mode.

In [ ]:
print(account.filter(lambda x:float(x[5])<1000).map(lambda x:(x[4],1)).reduceByKey(lambda a,b:a+b).collect())

## Q26. Premium High Risk

Find Premium customers classified as High risk.

In [ ]:
print(customer.filter(lambda x:x[3]=='Premium' and x[4]=='High').collect())

## Q27. Segment Counts

Count customers by customer segment.

In [ ]:
print(customer.map(lambda x:(x[3],1)).reduceByKey(lambda a,b:a+b).collect())

## Q28. City Customer Counts

Count customers by city.

In [ ]:
print(customer.map(lambda x:(x[2],1)).reduceByKey(lambda a,b:a+b).collect())

## Q29. Mode Maximum

Find maximum transaction by mode.

In [ ]:
print(account.map(lambda x:(x[4],float(x[5]))).reduceByKey(max).collect())

## Q30. Final Banking Report

Join customers and accounts and calculate total transaction value by customer segment.

In [ ]:
c=customer.map(lambda x:(x[0],x[3])); a=account.map(lambda x:(x[1],float(x[5]))); print(a.join(c).map(lambda x:(x[1][1],x[1][0])).reduceByKey(lambda a,b:a+b).sortBy(lambda x:x[1],ascending=False).collect())

## End of Project

The notebook is designed to run with the two CSV files in the same folder.